# 🎼 Mood-Based Music Generator — LSTM + MusicGen

No uploads needed. Just pick your settings and generate.

**Controls available:**
- 🎭 **Mood** — Happy / Sad / Energetic / Calm / Mysterious
- 🎹 **Instrument** — Piano / Strings / Guitar / Synth
- ⚡ **Tempo** — Slow / Medium / Fast
- 🎲 **Temperature** — how creative/random the output is
- 🎯 **Top-p** — nucleus sampling focus
- 🔢 **Note count** — how long the piece is
- 🎸 **MusicGen tokens** — length of AI audio output

In [ ]:
# Cell 1 — Install
!pip install pretty_midi transformers scipy gradio accelerate -q
!apt-get install fluidsynth -y -q
print('✅ Dependencies ready')

In [ ]:
# Cell 2 — Imports
import os, collections, pathlib, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
import pretty_midi
import gradio as gr
import scipy.io.wavfile as wavfile
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration
from IPython.display import Audio, display
print(f'TF {tf.__version__} | Torch {torch.__version__} | Gradio {gr.__version__}')
print(f'GPU: {"YES — " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

In [ ]:
# Cell 3 — Download MAESTRO dataset
!rm -rf dataset *.zip
!wget -q https://storage.googleapis.com/magentadata/datasets/maestro/v2.0.0/maestro-v2.0.0-midi.zip
!mkdir -p dataset
!unzip -q -o maestro-v2.0.0-midi.zip -d dataset/
data_dir  = pathlib.Path('dataset')
filenames = sorted(list(data_dir.glob('**/*.midi')) + list(data_dir.glob('**/*.mid')))
print(f'✅ {len(filenames)} MIDI files ready for training')

In [ ]:
# Cell 4 — MIDI helpers

# Instrument program numbers (General MIDI)
INSTRUMENT_MAP = {
    'Piano':   0,
    'Strings': 48,
    'Guitar':  25,
    'Synth':   81,
    'Vibraphone': 11,
}

def midi_to_notes(midi_file):
    pm = pretty_midi.PrettyMIDI(str(midi_file))
    if not pm.instruments:
        return pd.DataFrame()
    candidates = [i for i in pm.instruments if not i.is_drum] or pm.instruments
    instrument = max(candidates, key=lambda i: len(i.notes))
    if not instrument.notes:
        return pd.DataFrame()
    sorted_notes = sorted(instrument.notes, key=lambda n: n.start)
    notes = collections.defaultdict(list)
    prev_start = sorted_notes[0].start
    for note in sorted_notes:
        notes['pitch'].append(note.pitch)
        notes['step'].append(note.start - prev_start)
        notes['duration'].append(note.end - note.start)
        notes['velocity'].append(note.velocity)
        prev_start = note.start
    return pd.DataFrame(notes)


def notes_to_midi(notes_arr, output_file, tempo_bpm=120, instrument_name='Piano'):
    pm      = pretty_midi.PrettyMIDI(initial_tempo=tempo_bpm)
    program = INSTRUMENT_MAP.get(instrument_name, 0)
    inst    = pretty_midi.Instrument(program=program, name=instrument_name)
    t = 0.0
    for row in notes_arr:
        pitch    = int(np.clip(row[0], 21, 108))
        step     = max(0.0,  float(row[1]))
        duration = max(0.05, float(row[2]))
        velocity = int(np.clip(row[3] if len(row) > 3 else 80, 1, 127))
        t += step
        inst.notes.append(
            pretty_midi.Note(velocity=velocity, pitch=pitch, start=t, end=t+duration))
    pm.instruments.append(inst)
    pm.write(output_file)


def midi_to_wav(midi_path, wav_path):
    os.system(
        f'fluidsynth -ni /usr/share/sounds/sf2/FluidR3_GM.sf2 '
        f'"{midi_path}" -F "{wav_path}" -q')


print('✅ MIDI helpers defined')

In [ ]:
# Cell 5 — Load & normalise training data
NUM_FILES = 60
all_notes = []
for f in filenames[:NUM_FILES]:
    try:
        df = midi_to_notes(f)
        if len(df) >= 10:
            all_notes.append(df)
    except:
        continue
notes_df = pd.concat(all_notes, ignore_index=True)
print(f'Loaded {len(all_notes)} files → {len(notes_df):,} notes')

PITCH_MEAN = notes_df['pitch'].mean();    PITCH_STD = notes_df['pitch'].std()    + 1e-8
STEP_MEAN  = notes_df['step'].mean();     STEP_STD  = notes_df['step'].std()     + 1e-8
DUR_MEAN   = notes_df['duration'].mean(); DUR_STD   = notes_df['duration'].std() + 1e-8

def normalise_row(pitch, step, dur):
    return np.array([
        (pitch - PITCH_MEAN) / PITCH_STD,
        (step  - STEP_MEAN)  / STEP_STD,
        (dur   - DUR_MEAN)   / DUR_STD,
    ], dtype=np.float32)

def denormalise(p_n, s_n, d_n):
    return (
        p_n * PITCH_STD + PITCH_MEAN,
        s_n * STEP_STD  + STEP_MEAN,
        d_n * DUR_STD   + DUR_MEAN,
    )

def normalise_df(df):
    out = df.copy()
    out['pitch']    = (df['pitch']    - PITCH_MEAN) / PITCH_STD
    out['step']     = (df['step']     - STEP_MEAN)  / STEP_STD
    out['duration'] = (df['duration'] - DUR_MEAN)   / DUR_STD
    return out

notes_norm = normalise_df(notes_df)
print('✅ Normalisation done')

In [ ]:
# Cell 6 — Build sequences + pitch vocabulary
SEED_LENGTH   = 32
FEATURES      = ['pitch', 'step', 'duration']
PITCH_MIN     = 21
PITCH_CLASSES = 88

arr       = notes_norm[FEATURES].values.astype(np.float32)
raw_pitch = notes_df['pitch'].values.astype(np.int32)

X_seq, y_pitch_cls, y_timing = [], [], []
for i in range(len(arr) - SEED_LENGTH):
    X_seq.append(arr[i : i+SEED_LENGTH])
    y_pitch_cls.append(int(np.clip(raw_pitch[i+SEED_LENGTH], 21, 108)) - PITCH_MIN)
    y_timing.append(arr[i+SEED_LENGTH, 1:])

X_seq       = np.array(X_seq,       dtype=np.float32)
y_pitch_cls = np.array(y_pitch_cls, dtype=np.int32)
y_timing    = np.array(y_timing,    dtype=np.float32)
print(f'X {X_seq.shape} | pitch labels {y_pitch_cls.shape} | timing {y_timing.shape}')

In [ ]:
# Cell 7 — Bidirectional LSTM with separate pitch + timing heads
def build_model(seq_len=32, n_feat=3, n_pitch=88):
    inp = keras.Input(shape=(seq_len, n_feat), name='note_seq')
    x   = keras.layers.Bidirectional(
              keras.layers.LSTM(256, return_sequences=True), name='bilstm')(inp)
    x   = keras.layers.Dropout(0.3)(x)
    x   = keras.layers.LSTM(256, name='lstm')(x)
    x   = keras.layers.Dropout(0.3)(x)
    x   = keras.layers.Dense(128, activation='relu', name='shared')(x)
    pitch_out  = keras.layers.Dense(n_pitch, activation='softmax', name='pitch')(x)
    timing_out = keras.layers.Dense(2, name='timing')(x)
    m = keras.Model(inp, [pitch_out, timing_out], name='music_gen')
    m.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss={'pitch':'sparse_categorical_crossentropy', 'timing':'mse'},
        loss_weights={'pitch': 3.0, 'timing': 1.0},
        metrics={'pitch':'accuracy', 'timing':'mae'}
    )
    return m

model = build_model(SEED_LENGTH, len(FEATURES), PITCH_CLASSES)
model.summary()

In [ ]:
# Cell 8 — Train
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=1),
    keras.callbacks.ModelCheckpoint(
        'best_model.keras', save_best_only=True, monitor='val_loss'),
]
history = model.fit(
    X_seq, {'pitch': y_pitch_cls, 'timing': y_timing},
    epochs=40, batch_size=128, validation_split=0.1,
    callbacks=callbacks, verbose=1
)
print('✅ Training complete')

In [ ]:
# Cell 9 — Training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history.history['loss'],             label='train')
axes[0].plot(history.history['val_loss'],          label='val')
axes[0].set_title('Total Loss');       axes[0].legend()
axes[1].plot(history.history['pitch_accuracy'],     label='train')
axes[1].plot(history.history['val_pitch_accuracy'], label='val')
axes[1].set_title('Pitch Accuracy');   axes[1].legend()
axes[2].plot(history.history['timing_mae'],         label='train')
axes[2].plot(history.history['val_timing_mae'],     label='val')
axes[2].set_title('Timing MAE');       axes[2].legend()
for ax in axes: ax.set_xlabel('Epoch')
plt.tight_layout(); plt.show()

In [ ]:
# Cell 10 — Mood, tempo & instrument configuration

SCALES = {
    'Happy':      [0,2,4,5,7,9,11],   # major
    'Sad':        [0,2,3,5,7,8,10],   # natural minor
    'Energetic':  [0,2,3,5,7,9,10],   # dorian
    'Calm':       [0,2,4,7,9],         # pentatonic
    'Mysterious': [0,1,4,5,7,8,11],   # double harmonic
}

MOOD_VELOCITY = {
    'Happy':      (78, 108),
    'Sad':        (35,  60),
    'Energetic':  (90, 127),
    'Calm':       (40,  65),
    'Mysterious': (45,  80),
}

MOOD_PROMPTS = {
    'Happy':      'bright cheerful upbeat piano melody, major key, lively joyful',
    'Sad':        'slow melancholic minor key piano, soft emotional, adagio, tearful',
    'Energetic':  'fast driving energetic electronic music, powerful rhythm, high energy, intense',
    'Calm':       'gentle ambient piano, peaceful relaxing, slow soft dynamics, meditative',
    'Mysterious': 'dark mysterious cinematic music, eerie tension, haunting melody, suspenseful',
}

# Tempo BPM ranges per setting
TEMPO_MAP = {
    'Slow':   70,
    'Medium': 110,
    'Fast':   155,
}

# Timing multipliers per tempo (applied to step/duration)
TEMPO_STEP_SCALE = {
    'Slow':   1.6,
    'Medium': 1.0,
    'Fast':   0.55,
}

def snap_to_scale(pitch, scale, root=60):
    octave    = pitch // 12
    root_note = root % 12
    rel       = (pitch % 12 - root_note) % 12
    closest   = min(scale, key=lambda x: abs(x - rel))
    return int(np.clip(octave*12 + root_note + closest, 21, 108))


def apply_mood(pitch, step, dur, mood, tempo, temperature=1.0, prev_pitch=None):
    scale      = SCALES.get(mood, list(range(12)))
    step_scale = TEMPO_STEP_SCALE.get(tempo, 1.0)

    # mood-specific pitch and timing shaping
    if mood == 'Happy':
        pitch += int(np.random.choice([0,2,4]) * temperature)
        dur   *= 0.75
        step   = max(0.05, step * 0.85)

    elif mood == 'Sad':
        pitch -= int(np.random.choice([0,2,3]) * temperature)
        dur   *= 1.7
        step  *= 1.3

    elif mood == 'Energetic':
        pitch += int(np.random.choice([-7,-5,-2,2,5,7]) * temperature)
        step, dur = [(0.15,0.15),(0.2,0.2),(0.25,0.25),(0.25,0.5)][np.random.randint(4)]
        if prev_pitch and pitch == prev_pitch:
            pitch += int(np.random.choice([-3,3,5,-5]) * temperature)

    elif mood == 'Calm':
        dur  *= 2.0
        step  = max(0.15, step * 1.5)

    elif mood == 'Mysterious':
        pitch += int(np.random.choice([-1,0,1,5,-5]) * temperature)
        dur   *= 1.4
        step  *= np.random.choice([0.8, 1.0, 1.3])

    # apply tempo scaling
    step *= step_scale
    dur  *= step_scale

    pitch = snap_to_scale(int(np.clip(pitch, 21, 108)), scale)
    v_lo, v_hi = MOOD_VELOCITY.get(mood, (55, 95))
    vel = int(np.clip(np.random.randint(v_lo, v_hi+1), 1, 127))
    return pitch, max(0.0, float(step)), max(0.05, float(dur)), vel


print('✅ Mood + tempo + instrument config ready')

In [ ]:
# Cell 11 — Nucleus (top-p) sampling
def nucleus_sample(probs, temperature=1.0, top_p=0.9):
    p = np.array(probs, dtype=np.float64)
    p = np.power(p + 1e-10, 1.0 / max(temperature, 0.01))
    p /= p.sum()
    sorted_idx = np.argsort(p)[::-1]
    cumsum     = np.cumsum(p[sorted_idx])
    cutoff     = np.searchsorted(cumsum, top_p) + 1
    nucleus    = sorted_idx[:cutoff]
    np_nucleus = p[nucleus]; np_nucleus /= np_nucleus.sum()
    return int(np.random.choice(nucleus, p=np_nucleus))

print('✅ Nucleus sampling ready')

In [ ]:
# Cell 12 — LSTM music generator (starts from a random training seed)

def generate_lstm(mood, tempo, num_notes=200, temperature=1.0, top_p=0.9):
    """
    Pick a random seed from training data and generate num_notes notes.
    No file upload needed — seed comes from MAESTRO.
    """
    # pick a random seed window from training data
    seed_idx  = np.random.randint(0, len(X_seq))
    input_seq = X_seq[seed_idx].copy()

    generated  = []
    prev_pitch = None

    for _ in range(num_notes):
        pitch_probs, timing_pred = model.predict(
            input_seq[np.newaxis,...], verbose=0)

        # nucleus sample pitch
        pitch_cls = nucleus_sample(pitch_probs[0], temperature, top_p)
        pitch_r   = float(pitch_cls + PITCH_MIN)

        # timing with small noise
        t_pred        = timing_pred[0] + np.random.randn(2) * temperature * 0.1
        _, step_r, dur_r = denormalise(0, t_pred[0], t_pred[1])

        pitch, step, dur, vel = apply_mood(
            int(round(pitch_r)), float(step_r), float(dur_r),
            mood, tempo, temperature, prev_pitch)

        generated.append([pitch, step, dur, vel])
        prev_pitch = pitch
        input_seq  = np.vstack([input_seq[1:], normalise_row(pitch, step, dur)])

    return np.array(generated)


print('✅ LSTM generator ready')

In [ ]:
# Cell 13 — Load MusicGen from HuggingFace
MG_MODEL_ID = 'facebook/musicgen-small'
print(f'Loading {MG_MODEL_ID} ...')
mg_processor = AutoProcessor.from_pretrained(MG_MODEL_ID)
mg_model     = MusicgenForConditionalGeneration.from_pretrained(MG_MODEL_ID)
mg_device    = 'cuda' if torch.cuda.is_available() else 'cpu'
mg_model.to(mg_device)
mg_model.eval()
print(f'✅ MusicGen ready on {mg_device}')

In [ ]:
# Cell 14 — Piano roll visualisation helper
def piano_roll_from_midi(midi_path, title='Generated Music', cmap=plt.cm.plasma):
    pm   = pretty_midi.PrettyMIDI(str(midi_path))
    inst = max((i for i in pm.instruments if not i.is_drum),
               key=lambda i: len(i.notes), default=pm.instruments[0])
    fig, ax = plt.subplots(figsize=(13, 3))
    fig.patch.set_facecolor('#0d0d12')
    ax.set_facecolor('#0f0f1a')
    for note in inst.notes:
        c = cmap((note.pitch - 21) / 87)
        ax.barh(note.pitch, note.end - note.start,
                left=note.start, height=0.85, color=c, alpha=0.88)
    ax.set_xlabel('Time (s)', color='#aaa', fontsize=9)
    ax.set_ylabel('MIDI Pitch', color='#aaa', fontsize=9)
    ax.set_title(title, color='white', fontsize=11, pad=6)
    ax.tick_params(colors='#777', labelsize=8)
    for spine in ax.spines.values(): spine.set_edgecolor('#2a2a3a')
    plt.tight_layout()
    return fig

print('✅ Piano roll helper ready')

In [ ]:
# Cell 15 — Full generation pipeline
OUT_DIR = '/tmp/music_gen'
os.makedirs(OUT_DIR, exist_ok=True)

def run_pipeline(
    mood,
    tempo        = 'Medium',
    instrument   = 'Piano',
    num_notes    = 200,
    temperature  = 1.0,
    top_p        = 0.9,
    mg_tokens    = 512,
):
    tempo_bpm = TEMPO_MAP.get(tempo, 110)

    # ── Step 1: LSTM generation ───────────────────────────────────────────────
    print(f'[1/3] LSTM generating {num_notes} notes  mood={mood}  tempo={tempo}  instrument={instrument}')
    notes = generate_lstm(mood, tempo, num_notes, temperature, top_p)

    lstm_mid = f'{OUT_DIR}/lstm_{mood}_{tempo}.mid'
    lstm_wav = f'{OUT_DIR}/lstm_{mood}_{tempo}.wav'
    notes_to_midi(notes, lstm_mid, tempo_bpm=tempo_bpm, instrument_name=instrument)
    midi_to_wav(lstm_mid, lstm_wav)

    # ── Step 2: Piano roll ────────────────────────────────────────────────────
    print('[2/3] Rendering piano roll ...')
    roll_fig = piano_roll_from_midi(
        lstm_mid,
        title=f'{mood} — {instrument} — {tempo} tempo'
    )

    # pitch distribution chart
    pitch_counts = np.zeros(88)
    for row in notes:
        idx = int(np.clip(row[0], 21, 108)) - 21
        pitch_counts[idx] += 1
    dist_fig, ax = plt.subplots(figsize=(13, 2))
    dist_fig.patch.set_facecolor('#0d0d12')
    ax.set_facecolor('#0f0f1a')
    colors = plt.cm.plasma(np.linspace(0, 1, 88))
    ax.bar(range(88), pitch_counts, color=colors, width=1.0)
    ax.set_xlabel('Piano Key (0=A0, 87=C8)', color='#aaa', fontsize=9)
    ax.set_ylabel('Count', color='#aaa', fontsize=9)
    ax.set_title('Pitch Distribution', color='white', fontsize=10)
    ax.tick_params(colors='#777', labelsize=8)
    for spine in ax.spines.values(): spine.set_edgecolor('#2a2a3a')
    plt.tight_layout()

    # ── Step 3: MusicGen ──────────────────────────────────────────────────────
    print(f'[3/3] MusicGen generating audio ...')
    prompt = MOOD_PROMPTS.get(mood, 'expressive piano music')
    print(f'      Prompt: "{prompt}"')
    with torch.no_grad():
        inputs     = mg_processor(text=[prompt], padding=True,
                                  return_tensors='pt').to(mg_device)
        audio_vals = mg_model.generate(**inputs, max_new_tokens=mg_tokens)
    mg_wav   = f'{OUT_DIR}/musicgen_{mood}.wav'
    audio_np = audio_vals[0, 0].cpu().numpy().astype(np.float32)
    audio_np = audio_np / (np.abs(audio_np).max() + 1e-8)
    wavfile.write(mg_wav, rate=32000, data=(audio_np * 32767).astype(np.int16))

    info = (
        f'Generation summary\n'
        f'  Mood        : {mood}\n'
        f'  Tempo       : {tempo} ({tempo_bpm} BPM)\n'
        f'  Instrument  : {instrument}\n'
        f'  Notes       : {num_notes}\n'
        f'  Temperature : {temperature}\n'
        f'  Top-p       : {top_p}\n'
        f'  MusicGen    : {MG_MODEL_ID}\n'
        f'  Prompt      : {prompt}\n'
    )
    print('\n' + info)
    return lstm_wav, lstm_mid, mg_wav, roll_fig, dist_fig, info


print('✅ Pipeline ready')

In [ ]:
# Cell 16 — Quick test inside Colab (no Gradio needed)
lstm_wav, lstm_mid, mg_wav, roll_fig, dist_fig, info = run_pipeline(
    mood        = 'Happy',
    tempo       = 'Medium',
    instrument  = 'Piano',
    num_notes   = 150,
    temperature = 1.0,
    top_p       = 0.9,
    mg_tokens   = 512,
)

roll_fig.show()
dist_fig.show()
plt.show()

print('\n🎹  LSTM Piano:')
display(Audio(filename=lstm_wav))
print('\n🎸  MusicGen AI Audio:')
display(Audio(filename=mg_wav))

In [ ]:
# Cell 17 — Gradio Web App
# No file upload — just pick mood, tempo, instrument and generate

APP_CSS = (
    '.gradio-container { max-width: 980px !important; margin: 0 auto; }'
    'footer { display: none !important; }'
    'h1 { letter-spacing: -0.5px; }'
)

def gradio_run(mood, tempo, instrument, num_notes,
               temperature, top_p, mg_tokens):
    try:
        lstm_wav, lstm_mid, mg_wav, roll_fig, dist_fig, info = run_pipeline(
            mood        = mood,
            tempo       = tempo,
            instrument  = instrument,
            num_notes   = int(num_notes),
            temperature = float(temperature),
            top_p       = float(top_p),
            mg_tokens   = int(mg_tokens),
        )
        return lstm_wav, lstm_mid, mg_wav, roll_fig, dist_fig, info
    except Exception as e:
        import traceback
        return None, None, None, None, None, f'Error: {e}\n{traceback.format_exc()}'


with gr.Blocks(
    title='🎼 Music Generator',
    css=APP_CSS,
    theme=gr.themes.Base(
        primary_hue='violet',
        neutral_hue='slate',
        font=[gr.themes.GoogleFont('DM Sans'), 'sans-serif']
    )
) as demo:

    gr.Markdown(
        '# 🎼 Mood-Based Music Generator\n'
        'Choose your mood and settings — the AI composes original music for you.'
    )

    # ── Controls row ─────────────────────────────────────────────────────────
    with gr.Row():
        with gr.Column(scale=1):

            gr.Markdown('### 🎭 Mood')
            mood_radio = gr.Radio(
                choices=['Happy', 'Sad', 'Energetic', 'Calm', 'Mysterious'],
                value='Happy',
                label='',
                info='Shapes the scale, rhythm, and feel of the music'
            )

            gr.Markdown('### 🎹 Instrument')
            instrument_radio = gr.Radio(
                choices=['Piano', 'Strings', 'Guitar', 'Synth', 'Vibraphone'],
                value='Piano',
                label='',
                info='Changes the MIDI instrument timbre'
            )

            gr.Markdown('### ⚡ Tempo')
            tempo_radio = gr.Radio(
                choices=['Slow', 'Medium', 'Fast'],
                value='Medium',
                label='',
                info='Slow=70 BPM  |  Medium=110 BPM  |  Fast=155 BPM'
            )

        with gr.Column(scale=1):

            gr.Markdown('### 🔧 Fine-tune')
            num_notes_sl = gr.Slider(
                50, 400, value=200, step=10,
                label='Number of notes',
                info='More notes = longer piece'
            )
            temp_sl = gr.Slider(
                0.5, 1.8, value=1.0, step=0.05,
                label='Temperature',
                info='Low = safe & predictable  |  High = wild & creative'
            )
            top_p_sl = gr.Slider(
                0.5, 1.0, value=0.9, step=0.05,
                label='Top-p (nucleus sampling)',
                info='Lower = more focused note choices'
            )
            mg_tokens_sl = gr.Slider(
                128, 1024, value=512, step=128,
                label='MusicGen duration (tokens)',
                info='512 ≈ 10s  |  1024 ≈ 20s  (slower on CPU)'
            )

            gr.Markdown('&nbsp;')
            run_btn = gr.Button('🎵  Generate Music', variant='primary', size='lg')

    # ── Visuals row ───────────────────────────────────────────────────────────
    gr.Markdown('---')
    gr.Markdown('### 🎨 Visualisation')
    with gr.Row():
        roll_plot = gr.Plot(label='Piano Roll')
        dist_plot = gr.Plot(label='Pitch Distribution')

    # ── Summary ───────────────────────────────────────────────────────────────
    summary_box = gr.Textbox(
        label='Generation Summary', lines=9, interactive=False)

    # ── Audio outputs ─────────────────────────────────────────────────────────
    gr.Markdown('---')
    gr.Markdown('### 🔊 Listen & Download')
    with gr.Row():
        with gr.Column():
            gr.Markdown('**🎹 LSTM — Structured melody**')
            gr.Markdown('*Fast, offline. Note-by-note generation from the RNN.*')
            lstm_audio = gr.Audio(label='', type='filepath')
            lstm_dl    = gr.File(label='Download MIDI')
        with gr.Column():
            gr.Markdown('**🎸 MusicGen — AI polished audio**')
            gr.Markdown('*HuggingFace model. Richer sound, mood-guided prompt.*')
            mg_audio = gr.Audio(label='', type='filepath')

    # ── Wire up ───────────────────────────────────────────────────────────────
    run_btn.click(
        fn=gradio_run,
        inputs=[
            mood_radio, tempo_radio, instrument_radio,
            num_notes_sl, temp_sl, top_p_sl, mg_tokens_sl
        ],
        outputs=[lstm_audio, lstm_dl, mg_audio,
                 roll_plot, dist_plot, summary_box],
    )

demo.launch(share=True, debug=False)